In [1]:
import os
import json
import re
import base64
import httpx
from pathlib import Path
from notion_client import Client
from dotenv import load_dotenv

load_dotenv() # .env 파일 로드

# 1. 설정값 (.env 파일에서 읽어옵니다)
CLIENT_ID = os.getenv("NOTION_CLIENT_ID")
CLIENT_SECRET = os.getenv("NOTION_CLIENT_SECRET")
REDIRECT_URI = "http://localhost:8000/callback"

output_dir = Path("extracted_notion_data_v2")
output_dir.mkdir(exist_ok=True)
visited_nodes = set()

## Step 1: 인증 링크 생성
아래 코드를 실행하고 출력된 링크를 클릭하여 노션 권한을 승인하세요.

In [2]:
if not CLIENT_ID:
    print("❌ 오류: .env 파일에 NOTION_CLIENT_ID가 설정되지 않았습니다.")
else:
    auth_url = f"https://api.notion.com/v1/oauth/authorize?client_id={CLIENT_ID}&response_type=code&owner=user&redirect_uri={REDIRECT_URI}"
    print(f"🔗 아래 링크를 클릭하여 노션을 연동하세요:\n{auth_url}")

🔗 아래 링크를 클릭하여 노션을 연동하세요:
https://api.notion.com/v1/oauth/authorize?client_id=2efd872b-594c-8064-b5eb-003734e5089a&response_type=code&owner=user&redirect_uri=http://localhost:8000/callback


## Step 2: 인증 코드(Code) 입력 및 토큰 교환
링크 클릭 후 브라우저 주소창에 `code=...` 이라고 나오는 부분을 아래에 입력하세요.

In [ ]:
auth_code = input("브라우저 주소창에서 복사한 'code' 값을 입력하세요: ")

def get_access_token(code):
    creds = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()
    
    response = httpx.post(
        "https://api.notion.com/v1/oauth/token",
        json={
            "grant_type": "authorization_code",
            "code": code,
            "redirect_uri": REDIRECT_URI
        },
        headers={
            "Authorization": f"Basic {creds}",
            "Content-Type": "application/json"
        }
    )
    if response.status_code == 200:
        print("✅ 인증 성공!")
        return response.json().get("access_token")
    else:
        print(f"❌ 인증 실패: {response.text}")
        return None

if auth_code:
    access_token = get_access_token(auth_code)
    if access_token:
        notion = Client(auth=access_token, notion_version="2025-09-03")
    else:
        print("토큰을 가져오지 못했습니다.")
else:
    print("인증 코드가 입력되지 않았습니다.")

✅ 인증 성공!


## Step 3: 크롤링 엔진
데이터베이스 쿼리 방식이 바뀐 부분을 반영했습니다. 이미지 크롤링 기능이 추가되었습니다.

In [ ]:
def get_safe_name(name):
    if not name: return "Untitled"
    name = re.sub(r'[\\/:*?"<>|]', '_', name).strip(" . ")
    return name if name else "Untitled"

def get_all_blocks(parent_id):
    blocks = []
    has_more = True
    start_cursor = None
    while has_more:
        try:
            response = notion.blocks.children.list(block_id=parent_id, start_cursor=start_cursor)
            blocks.extend(response["results"])
            has_more = response["has_more"]
            start_cursor = response["next_cursor"]
        except:
            break
    return blocks

def download_image(url, save_path):
    try:
        with httpx.Client() as client:
            response = client.get(url)
            if response.status_code == 200:
                save_path.parent.mkdir(parents=True, exist_ok=True)
                with open(save_path, "wb") as f:
                    f.write(response.content)
                return True
    except Exception as e:
        print(f"    [!] 이미지 다운로드 실패: {e}")
    return False

def process_node(node_id, current_path, node_type="page"):
    if node_id in visited_nodes: return
    visited_nodes.add(node_id)
    
    try:
        if node_type == "page":
            p = notion.pages.retrieve(page_id=node_id)
            title = "Untitled"
            for prop in p.get("properties", {}).values():
                if prop["type"] == "title":
                    title = "".join([t["plain_text"] for t in prop.get("title", [])])
            
            safe_title = get_safe_name(title)
            print(f"📄 추출: {safe_title}")
            current_path.mkdir(parents=True, exist_ok=True)
            
            markdown = ""
            blocks = get_all_blocks(node_id)
            for block in blocks:
                b_type = block["type"]
                if b_type == "child_page":
                    process_node(block["id"], current_path / safe_title, "page")
                elif b_type == "child_database":
                    process_node(block["id"], current_path / safe_title, "database")
                elif b_type in ["paragraph", "heading_1", "heading_2", "heading_3", "bulleted_list_item"]:
                    rich_text = block[b_type].get("rich_text", [])
                    text = "".join([t["plain_text"] for t in rich_text])
                    if text: markdown += f"{text}\n\n"
                elif b_type == "image":
                    img_data = block["image"]
                    img_type = img_data["type"]
                    img_url = img_data[img_type].get("url")
                    
                    if img_url:
                        # 파일 확장자 추출 및 이름 생성
                        ext = ".png"
                        clean_url = img_url.split("?")[0] if "?" in img_url else img_url
                        for e in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
                            if clean_url.lower().endswith(e):
                                ext = e
                                break
                        
                        img_name = f"{node_id}_{block['id']}{ext}"
                        img_save_path = current_path / "images" / img_name
                        
                        if download_image(img_url, img_save_path):
                            markdown += f"![image](images/{img_name})\n\n"
                        else:
                            markdown += f"![image]({img_url})\n\n"
            
            with open(current_path / f"{safe_title}.md", "w", encoding="utf-8") as f:
                f.write(markdown)

        elif node_type == "database":
            try:
                db = notion.databases.retrieve(database_id=node_id)
                db_title = get_safe_name("".join([t["plain_text"] for t in db.get("title", [])]))
                print(f"🗃️ DB 발견: {db_title}")
                
                current_path.mkdir(parents=True, exist_ok=True)
                
                # 2025-09-03 API 호환성 처리
                if hasattr(notion, "data_sources"):
                    pages = notion.data_sources.query(data_source_id=node_id).get("results", [])
                else:
                    pages = notion.databases.query(database_id=node_id).get("results", [])
                
                for page in pages:
                    process_node(page["id"], current_path / db_title, "page")
            except Exception as e:
                print(f"    [!] DB 쿼리 건너뜀 (권한 부족 가능성): {e}")
    except Exception as e:
        print(f"    [!] 처리 실패: {e}")

def start_crawl():
    print("🚀 노션 데이터 수집 시작...")
    try:
        search_results = notion.search().get("results", [])
        for item in search_results:
            process_node(item["id"], output_dir, node_type=item["object"])
        print("✅ 완료!")
    except Exception as e:
        print(f"탐색 실패: {e}")

In [ ]:
if 'access_token' in globals() and access_token:
    start_crawl()

🚀 노션 데이터 수집 시작...
📄 추출: 123
✅ 완료!
